# Module 02: Experiment Tracking with MLflow

**What you'll learn:**
- Why experiment tracking matters (and why spreadsheets don't work)
- MLflow concepts: experiments, runs, parameters, metrics, artifacts
- How to log and compare training runs
- How to query and analyze experiments programmatically

**Time:** ~1.5 hours

## 1. The Problem: Lost Experiments

> "I trained a model last Tuesday that had great results. What hyperparameters did I use? Which dataset version? I can't remember."

Sound familiar? When you train models in notebooks, results disappear when you clear outputs or close the tab.

Common workarounds that **don't scale**:
- Spreadsheets → manual entry, gets outdated, no artifacts
- Naming files `model_v2_final_FINAL_v3.pkl` → chaos
- Print statements → lost when notebook restarts

**Experiment tracking** solves this by automatically recording every detail of every run.

## 2. What is MLflow?

MLflow is an open-source platform for managing the ML lifecycle. We'll focus on **MLflow Tracking**.

Key concepts:

| Concept | What it is | Example |
|---------|-----------|--------|
| **Experiment** | A named project/group of runs | `energy-demand-forecasting` |
| **Run** | One execution of training code | "XGBoost with lr=0.05" |
| **Parameters** | Inputs you chose | `learning_rate=0.05`, `max_depth=6` |
| **Metrics** | Results you measured | `rmse=4.5`, `mae=3.2` |
| **Artifacts** | Files you saved | model.pkl, predictions.png |
| **Tags** | Metadata labels | `model_type=xgboost`, `author=you` |

## 3. Your First MLflow Run

Let's log a simple experiment to understand the basics:

In [ ]:
import sys
sys.path.insert(0, '../src')
import mlflow
import numpy as np

# Use local file storage (no server needed for learning)
mlflow.set_tracking_uri('file:///tmp/mlflow_workshop')
mlflow.set_experiment('energy-demand-workshop')

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: energy-demand-workshop")
print("\nReady to log runs!")

In [ ]:
# Log your first run
with mlflow.start_run(run_name='my_first_run'):
    # Log parameters (the choices you made)
    mlflow.log_param('model_type', 'linear_regression')
    mlflow.log_param('alpha', 0.01)
    
    # Log metrics (the results)
    mlflow.log_metric('rmse', 15.3)
    mlflow.log_metric('mae', 12.1)
    mlflow.log_metric('r2', 0.82)
    
    # Log a tag (metadata)
    mlflow.set_tag('author', 'workshop_student')
    
    run_id = mlflow.active_run().info.run_id
    print(f'Run logged successfully!')
    print(f'Run ID: {run_id}')
    print(f'Parameters: model_type=linear_regression, alpha=0.01')
    print(f'Metrics: rmse=15.3, mae=12.1, r2=0.82')

That's it! MLflow saved everything — parameters, metrics, and metadata — automatically. Even if you close this notebook and come back next month, the run is still there.

## 4. Tracking Real Model Training

Now let's track **actual** model training. First, prepare our data:

In [ ]:
from energy_forecast.data.synthetic import SyntheticDataGenerator
from energy_forecast.features.engineering import FeatureEngineer
from energy_forecast.evaluation.metrics import MetricsCalculator
from sklearn.preprocessing import StandardScaler

# Generate and prepare data
gen = SyntheticDataGenerator(num_buildings=3, start_date='2023-01-01', end_date='2023-12-31', random_seed=42)
df = gen.generate()

engineer = FeatureEngineer()
df = engineer.create_time_features(df)
df = engineer.create_lag_features(df, 'energy_demand_kwh', [1, 2, 3, 24])
df = engineer.create_rolling_features(df, 'energy_demand_kwh', [6, 24])
df = engineer.create_weather_features(df)
df = engineer.create_calendar_features(df)
df = df.dropna()

feature_names = engineer.get_feature_names(df)
X = df[feature_names].values
y = df['energy_demand_kwh'].values

n = len(X)
X_train, y_train = X[:int(n*0.7)], y[:int(n*0.7)]
X_val, y_val = X[int(n*0.7):int(n*0.85)], y[int(n*0.7):int(n*0.85)]
X_test, y_test = X[int(n*0.85):], y[int(n*0.85):]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

### Train a Linear Model with MLflow Tracking

In [ ]:
from energy_forecast.models.linear import LinearForecaster

with mlflow.start_run(run_name='linear_baseline'):
    model = LinearForecaster(model_type='ridge', alpha=1.0)
    mlflow.log_params(model.get_params())
    
    model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    predictions = model.predict(X_test)
    metrics = MetricsCalculator.compute_all(y_test, predictions)
    
    for name, value in metrics.items():
        mlflow.log_metric(name, value)
    mlflow.set_tag('model_type', 'linear')
    
    print(f"Linear Model Results:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

### Now Train XGBoost with Different Configurations

This is where tracking shines — let's compare multiple configurations:

In [ ]:
from energy_forecast.models.xgboost_model import XGBoostForecaster

for n_est in [100, 300, 500]:
    with mlflow.start_run(run_name=f'xgboost_n{n_est}'):
        model = XGBoostForecaster(n_estimators=n_est, max_depth=6, learning_rate=0.05)
        mlflow.log_params(model.get_params())
        
        model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
        predictions = model.predict(X_test)
        metrics = MetricsCalculator.compute_all(y_test, predictions)
        
        for name, value in metrics.items():
            mlflow.log_metric(name, value)
        mlflow.set_tag('model_type', 'xgboost')
        
        print(f"XGBoost (n_estimators={n_est}): RMSE={metrics['rmse']:.4f}, R2={metrics['r2']:.4f}")

## 5. Comparing Experiments

This is the **killer feature** of experiment tracking — comparing all your runs side by side:

In [ ]:
from mlflow.tracking import MlflowClient
import matplotlib.pyplot as plt

client = MlflowClient()
experiment = client.get_experiment_by_name('energy-demand-workshop')
runs = client.search_runs(experiment.experiment_id, order_by=['metrics.rmse ASC'])

print(f"{'Run Name':<25} {'Model':<10} {'RMSE':<10} {'MAE':<10} {'R2':<10}")
print('-' * 65)
names, rmses = [], []
for run in runs:
    name = run.info.run_name or 'unnamed'
    model_type = run.data.tags.get('model_type', '?')
    rmse = run.data.metrics.get('rmse', 0)
    mae = run.data.metrics.get('mae', 0)
    r2 = run.data.metrics.get('r2', 0)
    if rmse > 0:
        print(f"{name:<25} {model_type:<10} {rmse:<10.4f} {mae:<10.4f} {r2:<10.4f}")
        names.append(name); rmses.append(rmse)

# Visualize
if names:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['coral' if 'linear' in n else 'steelblue' for n in names]
    ax.barh(names, rmses, color=colors)
    ax.set_xlabel('RMSE (lower is better)')
    ax.set_title('Model Comparison')
    plt.tight_layout()
    plt.show()

## 6. Logging Artifacts

Artifacts are files saved alongside your run — model binaries, plots, data samples. Let's log a prediction plot:

In [ ]:
import matplotlib.pyplot as plt

with mlflow.start_run(run_name='xgboost_with_plot'):
    model = XGBoostForecaster(n_estimators=500, max_depth=6, learning_rate=0.05)
    mlflow.log_params(model.get_params())
    model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    predictions = model.predict(X_test)
    metrics = MetricsCalculator.compute_all(y_test, predictions)
    for name, value in metrics.items():
        mlflow.log_metric(name, value)
    
    # Create and log a plot as an artifact
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(y_test[:200], label='Actual', alpha=0.8)
    ax.plot(predictions[:200], label='Predicted', alpha=0.8)
    ax.legend()
    ax.set_title('Actual vs Predicted Energy Demand')
    ax.set_ylabel('kWh')
    fig.savefig('/tmp/predictions.png', dpi=100)
    mlflow.log_artifact('/tmp/predictions.png')
    plt.show()
    print('Plot saved as MLflow artifact!')

## 7. Exercises

1. **More experiments**: Train 3 XGBoost models with different `max_depth` values (3, 6, 9) and log each to MLflow. Which depth works best?

2. **Log feature importance**: After training XGBoost, create a feature importance bar chart and log it as an artifact.

3. **Find the best run**: Write code to programmatically find the run with the lowest RMSE and print all its parameters.

## 8. Key Takeaways

- **Always track experiments** — your future self will thank you
- MLflow gives you: reproducibility, comparison, collaboration
- **Parameters** = what you chose, **Metrics** = what you got, **Artifacts** = files you saved
- In production, MLflow runs on a server (with Docker Compose: `http://localhost:5000`)
- Query runs programmatically to automate model selection

**Next: [Notebook 03 - Model Development](./03_model_development.ipynb)**